# Comparer les modèles avec CLIP

Ce notebook compare des images produites **au préalable** à partir des mêmes prompts. Il ne génère pas les images et ne contacte aucun service de génération. Les images peuvent venir d’Iris-3B sans LoRA, d’Iris-3B avec LoRA et de tout autre modèle, local ou externe.

La mesure est une similarité image–texte : `100 × max(0, cosinus CLIP)` ; elle est affichée sur 100 pour faciliter la lecture. Ce n’est pas une note de qualité absolue ni une reproduction exacte du CLIPScore académique. CLIP n’est pas spécialisé dans le pixel art : vérifie aussi les sprites visuellement.


## 1. Préparer le test

Les prompts de [`prompts.jsonl`](prompts.jsonl) sont fixés et en anglais. Garde exactement le même texte pour tous les modèles : n’ajoute les tags propres au LoRA que dans son prompt de génération, et évalue toujours le prompt commun. Les graines aident à organiser les essais ; elles ne garantissent pas la même image ni le même bruit entre modèles.

Le kernel a besoin de `torch`, `transformers`, `Pillow`, `numpy`, `pandas` et `matplotlib`. Le modèle CLIP sera téléchargé au premier calcul. Le calcul utilise le CPU par défaut pour laisser libre le GPU distant qui annote le dataset.


In [ ]:
import sys, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import torch, transformers
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
EVAL_DIR = next((p for base in (cwd, *cwd.parents) for p in (base, base / "clip_score", base / "evaluating/clip_score", base / "notebook/evaluating/clip_score") if (p / "clip_evaluation.py").is_file()), None)
if EVAL_DIR is None:
    raise FileNotFoundError("Ouvre le notebook depuis le projet iris-pixelart.")
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))
from clip_evaluation import load_prompts, write_manifest_template, load_manifest, score_images, compare_by_prompt, save_csv

PROMPTS_PATH = EVAL_DIR / "prompts.jsonl"
WORK_DIR = EVAL_DIR / "work"  # Images et résultats locaux, ignorés par Git.
MANIFEST_PATH = WORK_DIR / "manifest.csv"
MODELS = ["iris_base", "iris_lora", "autre_modele"]  # Remplacer ou ajouter des modèles.
SEEDS = [42, 123, 2024]
BASELINE = "iris_base"
CLIP_MODEL = "openai/clip-vit-base-patch32"
DEVICE = "cpu"  # Mettre "cuda" uniquement si un GPU est disponible pour cette évaluation.
BATCH_SIZE = 8
prompts = load_prompts(PROMPTS_PATH)
print(f"{len(prompts)} prompts · {len(MODELS)} modèles · {len(SEEDS)} essais par prompt")


## 2. Préparer le manifeste

Exécute cette cellule une fois : elle crée `work/manifest.csv` sans écraser un fichier existant. Chaque ligne associe un modèle, un prompt, une graine et un chemin d’image relatif au manifeste. Génère ou copie les images à ces chemins, ou modifie les chemins dans le CSV. Chaque modèle doit couvrir les mêmes couples prompt/graine.

Laisse le LoRA activé uniquement pour `iris_lora`. Pour chaque modèle, conserve les réglages de génération, sa version, ses dimensions et sa graine dans tes notes : CLIP évalue les fichiers fournis et ne peut pas confirmer comment ils ont été produits.


In [ ]:
if MANIFEST_PATH.exists():
    print(f"Manifeste existant conservé : {MANIFEST_PATH}")
else:
    print("Manifeste créé :", write_manifest_template(MANIFEST_PATH, prompts, MODELS, SEEDS))
display(pd.read_csv(MANIFEST_PATH).head(12))


## 3. Vérifier les images

La validation refuse les fichiers manquants, les doublons et les comparaisons incomplètes. Elle ne devine pas si les images ont été générées avec le bon prompt ou les bons réglages.


In [ ]:
rows = load_manifest(MANIFEST_PATH, prompts, BASELINE)
display(pd.DataFrame(rows).groupby("model_id").size().rename("images").to_frame())
print(f"Comparaison complète : {len(rows)} images, {len(prompts)} prompts, référence {BASELINE}.")


## 4. Calculer les scores

Les images transparentes sont compositées sur blanc avant le prétraitement standard de CLIP. Son redimensionnement peut lisser les pixels ; il est identique pour toutes les images. Le score mesure l’accord avec le prompt commun, pas la netteté, la transparence ou la fidélité aux pixels.


In [ ]:
scores = score_images(rows, prompts, model_name=CLIP_MODEL, batch_size=BATCH_SIZE, device=DEVICE)
summary = compare_by_prompt(scores, BASELINE)
score_df = pd.DataFrame(scores)
summary_df = pd.DataFrame(summary).sort_values("mean_clip_similarity_0_100", ascending=False)
display(summary_df.round(2))


## 5. Comparer et sauvegarder

Chaque prompt a le même poids : on moyenne d’abord les graines d’un prompt, puis les prompts. L’intervalle à 95 % est un bootstrap apparié des différences par prompt ; avec peu de prompts il reste indicatif. Les CSV détaillés et la configuration sont enregistrés dans un nouveau dossier horodaté.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, max(3, len(summary_df) * 0.7)))
axes[0].barh(summary_df["model_id"], summary_df["mean_clip_similarity_0_100"], color="#6366f1")
axes[0].set_xlabel("Similarité CLIP moyenne / 100")
axes[0].invert_yaxis()
axes[0].set_title("Accord image–prompt")
for i, row in enumerate(summary_df.itertuples()):
    axes[1].plot([row.delta_ci_95_low, row.delta_ci_95_high], [i, i], color="#475569", linewidth=2)
    axes[1].scatter(row.delta_vs_baseline, i, color="#4f46e5", zorder=3)
axes[1].axvline(0, color="#94a3b8", linestyle="--")
axes[1].set_yticks(range(len(summary_df)), summary_df["model_id"])
axes[1].invert_yaxis()
axes[1].set_xlabel(f"Différence avec {BASELINE} · intervalle bootstrap 95 %")
axes[1].set_title("Comparaison appariée par prompt")
plt.tight_layout()
plt.show()

run_dir = WORK_DIR / "results" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
save_csv(run_dir / "scores.csv", scores)
save_csv(run_dir / "comparison.csv", summary)
config = {"clip_model": CLIP_MODEL, "clip_revision": scores[0]["clip_model_revision"],
          "torch_version": torch.__version__, "transformers_version": transformers.__version__, "metric": "100 * max(0, cosine)", "baseline": BASELINE,
          "device": DEVICE, "batch_size": BATCH_SIZE, "bootstrap_samples": 2000,
          "bootstrap_seed": 42, "background": "white", "prompt_sha256": hashlib.sha256(PROMPTS_PATH.read_bytes()).hexdigest(),
          "manifest_sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
          "created_at_utc": datetime.now(timezone.utc).isoformat()}
(run_dir / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Résultats enregistrés dans", run_dir)
